# 01 – EDA: Crack Detection Dataset (CRACK500 + DeepCrack)

**CSE445 – Road Damage Detection & Lane Segmentation**

This notebook performs Exploratory Data Analysis on the crack detection dataset.

Goals:
- Confirm image / mask pairs are correctly paired
- Visualise sample image–mask overlays
- Analyse resolution distribution
- Quantify class imbalance (crack vs background pixels)
- Preview augmented training samples

In [ ]:
# ── Environment setup ────────────────────────────────────────────────────
import sys, os

# For Google Colab: mount Drive and set RUN_ENV
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Clone / pull repo if needed
    REPO = '/content/drive/MyDrive/Road_Damage_Project'
    os.environ['RUN_ENV'] = 'colab'
except ImportError:
    # Running locally
    REPO = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    os.environ['RUN_ENV'] = 'local'

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('Repo root:', REPO)

In [ ]:
# Install dependencies (only needed on Colab)
# !pip install -q albumentations opencv-python-headless tqdm

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import config
from src.crack.preprocess import load_pairs, report_stats, split_dataset
from src.shared.transforms import get_train_transforms
from src.shared.dataset import SegmentationDataset

print('Config loaded. Crack image dir:', config.CRACK_IMG_DIR)

## 1. Load and Validate Image–Mask Pairs

In [ ]:
pairs = load_pairs()
print(f'Total pairs: {len(pairs)}')
# Show first 5
for img_p, mask_p in pairs[:5]:
    print(f'  {img_p.name:40s} ↔  {mask_p.name}')

## 2. Sample Image–Mask Overlays

In [ ]:
N = 6  # number of samples to display
fig, axes = plt.subplots(N, 3, figsize=(12, N * 3.5))
fig.suptitle('CRACK500 – Image / Mask / Overlay', fontsize=14, fontweight='bold')

for i, (img_path, mask_path) in enumerate(pairs[:N]):
    img  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    # Overlay: highlight crack pixels in red
    overlay = img.copy()
    crack_pixels = mask > 127
    overlay[crack_pixels] = [220, 30, 30]

    axes[i, 0].imshow(img);         axes[i, 0].set_title('Image')
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('Mask (GT)')
    axes[i, 2].imshow(overlay);    axes[i, 2].set_title('Overlay')

    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig('eda_crack_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: eda_crack_samples.png')

## 3. Resolution Distribution

In [ ]:
heights, widths = [], []
for img_path, _ in pairs:
    img = cv2.imread(str(img_path))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h); widths.append(w)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Image Height Distribution'); axes[0].set_xlabel('Height (px)')
axes[1].hist(widths,  bins=20, color='coral',    edgecolor='white')
axes[1].set_title('Image Width Distribution');  axes[1].set_xlabel('Width (px)')

print(f'Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')
print(f'Width : min={min(widths)},  max={max(widths)},  mean={np.mean(widths):.0f}')
plt.tight_layout(); plt.show()

## 4. Class Imbalance Analysis

In [ ]:
fg_ratios = []
for _, mask_path in pairs:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is not None:
        fg_ratios.append(float((mask > 127).mean()))

mean_fg = np.mean(fg_ratios)
print(f'Mean crack pixel ratio : {mean_fg:.4%}')
print(f'Mean background ratio  : {1-mean_fg:.4%}')
print(f'Imbalance ratio (bg:fg): {(1-mean_fg)/mean_fg:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(fg_ratios, bins=40, color='crimson', edgecolor='white')
axes[0].set_title('Distribution of Crack Pixel Ratio per Image')
axes[0].set_xlabel('Fraction of crack pixels')
axes[0].axvline(mean_fg, color='black', linestyle='--', label=f'Mean={mean_fg:.3%}')
axes[0].legend()

# Pie chart
axes[1].pie([1-mean_fg, mean_fg],
            labels=['Background', 'Crack'],
            colors=['#4e9af1', '#e8505b'],
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Overall Pixel Class Distribution')
plt.tight_layout(); plt.show()

## 5. Dataset Split Preview

In [ ]:
splits = split_dataset(pairs)
labels = list(splits.keys())
counts = [len(v) for v in splits.values()]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts, color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='white')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(cnt),
            ha='center', fontweight='bold')
ax.set_title('Dataset Split Sizes (70 / 15 / 15)')
ax.set_ylabel('Number of image-mask pairs')
plt.tight_layout(); plt.show()

## 6. Augmented Sample Preview

In [ ]:
# Save splits first (needed for SegmentationDataset)
from src.crack.preprocess import save_splits
save_splits(splits)

# Load training dataset with augmentation
train_csv = config.CRACK_SPLIT_DIR / 'train.csv'
train_tf  = get_train_transforms()
ds        = SegmentationDataset(train_csv, transform=train_tf)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Augmented Training Samples (each row = same image, different augmentation)',
             fontsize=12)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

idx = 0
for col in range(4):
    img_t, mask_t = ds[idx]
    # Denormalise
    img_np = (img_t.permute(1,2,0).numpy() * std + mean).clip(0,1)
    mask_np = mask_t.squeeze().numpy()
    axes[0, col].imshow(img_np);           axes[0, col].set_title(f'Aug Image #{col+1}')
    axes[1, col].imshow(mask_np, cmap='gray'); axes[1, col].set_title('Aug Mask')
    overlay = (img_np * 255).astype(np.uint8).copy()
    overlay[mask_np > 0.5] = [220, 30, 30]
    axes[2, col].imshow(overlay);          axes[2, col].set_title('Overlay')

for ax in axes.flatten(): ax.axis('off')
plt.tight_layout(); plt.show()

## Summary

- ✅ Loaded and validated image–mask pairs
- ✅ Confirmed class imbalance (crack pixels ≈ 2–5% of total)
- ✅ Checked resolution distribution
- ✅ Split into train / val / test (70 / 15 / 15)
- ✅ Verified augmentation pipeline works

**Next**: `02_EDA_lane.ipynb` → lane dataset EDA